```bash
CUDA_VISIBLE_DEVICES=7 vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8084 \
    --gpu-memory-utilization 0.85 \
    --enable-prefix-caching \
    --dtype bfloat16 \
    --max_model_len 32000 \
    --trust-remote-code
```

```bash
CUDA_VISIBLE_DEVICES=6 vllm serve Skywork/Skywork-o1-Open-PRM-Qwen-2.5-7B \
    --host 0.0.0.0 \
    --port 8082 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

```bash
CUDA_VISIBLE_DEVICES=5 vllm serve Qwen/Qwen2.5-Math-PRM-7B \
    --host 0.0.0.0 \
    --port 8083 \
    --gpu-memory-utilization 0.8 \
    --enable-prefix-caching \
    --dtype auto \
    --task reward
```

In [3]:
# from openai import OpenAI

# OPENAI_API_KEY = "EMPTY"
# OPENAI_API_BASE = "http://localhost:{PORT}/v1"

# causal_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8084),
# )
# causal_model = causal_client.models.list().data[0].id

# skywork_prm_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8082),
# )
# skywork_prm_model = skywork_prm_client.models.list().data[0].id

# qwen_prm_client = OpenAI(
#     api_key=OPENAI_API_KEY,
#     base_url=OPENAI_API_BASE.format(PORT=8083),
# )
# qwen_prm_model = qwen_prm_client.models.list().data[0].id


In [100]:
# def attack(df_sample, 
#     task_text, 
#     experiment_name,
#     use_augment_question_prm=True,
#     prm_clients = [skywork_prm_client, qwen_prm_client],
#     causal_client=causal_client
# ):
#     causal_model = causal_client.models.list().data[0].id
    

#     # Apply the augmentor function to each row in the DataFrame
#     aug_results = augmentor(df_sample, task_text, client=causal_client, model=causal_model)
#     df_aug = pd.DataFrame(aug_results)

#     # Check equivalence
#     equivalence_results = equivalence_check(df_sample, df_aug, client=causal_client, model=causal_model)
#     df_aug["equivalence"] = equivalence_results["equivalence"]
#     df_aug["body_equivalence_results"] = equivalence_results["body_equivalence_results"]

#     # PRM Scorer
#     for prm_client, prm_model in zip(prm_clients, prm_models):
#         rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
#                             steps=df["aug_steps"].tolist(), 
#                             client=prm_client, model=prm_model)
#         df[f"{prm_model}--aug_rewards"] = rewards
            
#     # concat the original and augmented DataFrames
#     for key in df_aug.keys():
#         df_sample[key] = df_aug[key]
    
#     # Save the DataFrame to a CSV file
#     df_sample.to_parquet(f"experiments/{experiment_name}.parquet", index=False)
#     return df_sample

In [4]:
import os
import pandas as pd
from constants.prompts_constants import (
    VERBOSE_TASK, CONSISE_TASK, EQ_TO_TEXT_TASK, CHANGE_NUMBERS_TASK
)
from utils.attack_utils import chatgpt_batch_augmentor, chatgpt_batch_equivalence_checker, prm_scorer

def attack_chatgpt_batch(df,
    task_text, 
    experiment_name,
    prm_clients=None,
    run_augmentor=True,
    run_equivalence_check=True,
    run_prm_scorer=True,
    use_augment_question_prm=False,

):
    """
    Run the attack on the given dataframe using the chatgpt batch API.
    Args:
        df: The dataframe to attack. Should have the following columns:
            - problem: The problem to attack.
            - steps: The steps to attack.
        task_text: The task to use for the attack.
        experiment_name: The name of the experiment.
        prm_clients: The prm clients to use for the attack.
        run_augmentor: Whether to run the augmentor.
        run_equivalence_check: Whether to run the equivalence check.
        run_prm_scorer: Whether to run the prm scorer.
        use_augment_question_prm: Whether to use the augmented question for the prm scorer.
    Returns:
        df: The dataframe with the attack results. 
            For each row, it will have the following columns:
                - problem: The problem to attack.
                - steps: The steps to attack.
                - aug_problem: The augmented question.
                - aug_steps: The augmented steps.
                - equivalence: Whether the augmented question and steps are equivalent to the original question and steps.
                - body_equivalence_results: The body of the equivalence check.
                - {prm_model}--aug_rewards: The rewards from the prm scorer for the augmented question and steps.
    """
    experiment_path = os.path.join("experiments", experiment_name)
    os.makedirs(experiment_path, exist_ok=True)
    if run_augmentor:
        df = chatgpt_batch_augmentor(df, task_text, experiment_path)
    if run_equivalence_check:
        df = chatgpt_batch_equivalence_checker(df, experiment_path)
    if run_prm_scorer:
        prm_models = [prm_client.models.list().data[0].id for prm_client in prm_clients]
        for prm_client, prm_model in zip(prm_clients, prm_models):
            rewards = prm_scorer(questions=df["aug_problem"].tolist() if use_augment_question_prm else df["problem"].tolist(),
                                steps=df["aug_steps"].tolist(), 
                                client=prm_client, model=prm_model)
            df[f"{prm_model}--aug_rewards"] = rewards
    
    df.to_parquet(os.path.join(experiment_path, "attack.parquet"), index=False)
    return df

In [5]:
df = pd.read_parquet("data/processbench.parquet")


## TODO: Filter dataset
sample_size = 100
df_sample = df.sample(sample_size, random_state=42).reset_index(drop=True)


In [6]:
aug_df = attack_chatgpt_batch(df_sample, 
                              task_text=VERBOSE_TASK,
                              experiment_name="chatgpt_batch_verbose_sanity_check",
                              prm_clients=None,
                              run_augmentor=True,
                              run_equivalence_check=True,
                              run_prm_scorer=False,
                              use_augment_question_prm=False
                            )

2025-05-03 13:39:40.900165 validating
2025-05-03 13:40:42.031274 validating
2025-05-03 13:41:43.005053 validating
2025-05-03 13:42:44.193849 in_progress
2025-05-03 13:43:45.220578 in_progress
2025-05-03 13:44:46.356176 in_progress
2025-05-03 13:45:47.484058 finalizing
2025-05-03 13:46:48.488174 completed
2025-05-03 13:47:58.246650 validating
2025-05-03 13:48:59.379134 validating
2025-05-03 13:50:00.510904 validating
2025-05-03 13:51:01.643392 validating
2025-05-03 13:52:02.776639 in_progress
2025-05-03 13:53:03.908430 in_progress
2025-05-03 13:54:04.835924 in_progress
2025-05-03 13:55:05.866275 in_progress
2025-05-03 13:56:06.895963 in_progress
2025-05-03 13:57:08.131815 finalizing
2025-05-03 13:58:09.091873 completed


In [2]:
import pandas as pd
aug_df = pd.read_parquet("experiments/chatgpt_batch_verbose_sanity_check/attack.parquet")

In [10]:
print("Problem:", aug_df["problem"].iloc[2])
print("Aug Problem:", aug_df["aug_problem"].iloc[2])
print("--------------------------------")
print("Steps:", aug_df["steps"].iloc[2])
print("Aug Steps:", aug_df["aug_steps"].iloc[2])


Problem: In ARMLopolis, every house number is a positive integer, and City Hall's address is 0. However, due to the curved nature of the cowpaths that eventually became the streets of ARMLopolis, the distance $d(n)$ between house $n$ and City Hall is not simply the value of $n$. Instead, if $n=3^{k} n^{\prime}$, where $k \geq 0$ is an integer and $n^{\prime}$ is an integer not divisible by 3 , then $d(n)=3^{-k}$. For example, $d(18)=1 / 9$ and $d(17)=1$. Notice that even though no houses have negative numbers, $d(n)$ is well-defined for negative values of $n$. For example, $d(-33)=1 / 3$ because $-33=3^{1} \cdot-11$. By definition, $d(0)=0$. Following the dictum "location, location, location," this Power Question will refer to "houses" and "house numbers" interchangeably.

Curiously, the arrangement of the houses is such that the distance from house $n$ to house $m$, written $d(m, n)$, is simply $d(m-n)$. For example, $d(3,4)=d(-1)=1$ because $-1=3^{0} \cdot-1$. In particular, if $m=n$

In [8]:
aug_df

,id,generator,problem,steps,final_answer_correct,label,split,steps_len,per_step_len,Qwen2.5-Math-PRM-7B,Skywork-o1-Open-PRM-Qwen-2.5-7B,aug_problem,aug_steps,equivalence,body_equivalence_results
0,gsm8k-291,Llama-3.1-70B-Instruct,Erin has 7 lollipops. Her mother gives Erin an...,[Let's break down the problem step by step: Fi...,True,-1,gsm8k,4,"[80, 210, 197, 46]","[1.0, 1.0, 1.0, 1.0]","[0.983596967483837, 0.9678992932829918, 0.9825...",Erin has 7 lollipops. Her mother gives Erin an...,[Let's break down the problem step by step: Fi...,True,<step_count>Y</step_count>\n <question>Y</que...
1,math-355,Qwen2.5-Math-72B-Instruct,There is a unique polynomial $P(x)$ of degree ...,[To find the polynomial \( P(x) \) of degree 8...,False,4,math,6,"[410, 504, 255, 197, 380, 50]","[0.95703125, 0.953125, 0.97265625, 1.0, 0.0247...","[0.23091976292927177, 0.17667160102063315, 0.1...",Given a polynomial \(P(x)\) with rational coef...,[To determine the polynomial \(P(x)\) with the...,True,<step_count>Y</step_count>\n <question>Y</que...
2,olympiadbench-696,Qwen2-7B-Instruct,"In ARMLopolis, every house number is a positiv...",[To determine which house(s) with a positive n...,True,1,olympiadbench,7,"[405, 330, 319, 137, 422, 115, 388]","[0.9609375, 0.70703125, 0.77734375, 0.98046875...","[0.23091976292927177, 0.21337717541660495, 0.4...","In the city of ARMLopolis, every house is assi...",[To determine the house(s) with a positive num...,True,<step_count>Y</step_count>\n <question>Y</que...
3,math-32,Meta-Llama-3-8B-Instruct,What is the following value when expressed as ...,"[To solve this problem, we can use the formula...",False,2,math,5,"[181, 267, 469, 212, 155]","[0.7578125, 0.083984375, 0.0233154296875, 0.48...","[0.45834870020339913, 0.2643583582610753, 0.20...",What is the value of the following sum when ex...,"[To solve for the value of this sum, we will u...",True,<step_count>Y</step_count>\n <question>Y</que...
4,math-79,Qwen2.5-Math-72B-Instruct,Let $f(z)= \frac{z+a}{z+b}$ and $g(z)=f(f(z))$...,"[To solve the problem, we start by analyzing t...",False,3,math,11,"[241, 310, 531, 128, 84, 227, 115, 234, 127, 7...","[0.796875, 0.98046875, 0.7109375, 0.14453125, ...","[0.22678363486528472, 0.3225259830643727, 0.29...",Let \( f(z)= \frac{z+a}{z+b} \) be a complex r...,"[To approach the solution, we first analyze th...",True,<step_count>Y</step_count>\n <question>Y</que...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,math-394,Qwen2-7B-Instruct,"Below are the graphs of two functions, $f(x)$ ...","[To find the largest value of \(f(x) - g(x)\),...",False,0,math,9,"[274, 51, 280, 314, 305, 359, 473, 493, 281]","[0.82421875, 0.98828125, 0.73828125, 0.8398437...","[0.17781086947804958, 0.19806417366892506, 0.1...","The graphs of two functions, \( f(x) \) and \(...",[To find the maximum value of \( f(x) - g(x) \...,True,<step_count>Y</step_count>\n <question>Y</que...
96,olympiadbench-902,Llama-3.1-70B-Instruct,Let $T=19$. Let $\mathcal{S}_{1}$ denote the a...,[To find the $T^{th}$ smallest number that occ...,True,-1,olympiadbench,8,"[409, 237, 116, 115, 262, 276, 72, 80]","[0.94921875, 0.82421875, 0.99609375, 1.0, 0.98...","[0.1412814160007008, 0.19314738002256465, 0.27...",Let \( T = 19 \). Define the arithmetic sequen...,[To identify the \( T \)-th smallest number co...,True,<step_count>Y</step_count>\n <question>Y</que...
97,omnimath-261,Qwen2.5-Math-72B-Instruct,"Find all positive integers $x,y,z$ and $t$ suc...",[To solve the equation \(2^x 3^y + 5^z = 7^t\)...,False,18,omnimath,19,"[819, 149, 148, 141, 66, 481, 200, 202, 192, 1...","[0.41015625, 0.99609375, 1.0, 0.99609375, 0.99...","[0.2173375210470625, 0.22405544923169474, 0.24...","Determine all positive integers \(x,y,z,\) and...",[To solve the given equation \(2^x 3^y + 5^z =...,True,<step_count>Y</step_count>\n <question>Y</que...
98,math-418,Qwen2-7B-Instruct,Let $S$ be the union of the set of all points ...,"[To solve this problem, let's break it down in..."